In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.proportion import proportions_ztest
import scikit_posthocs as sp
import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [3]:
portfolio = pd.read_csv('../data/portfolio.csv')
profile = pd.read_csv('../data/profile.csv')
transcript = pd.read_csv('../data/transcript.csv')

merged = pd.read_csv('../data/all_merged.csv')
promotion = pd.read_csv('../data/promotion_df.csv')
normal = pd.read_csv('../data/normal_flow_df.csv')
final = pd.read_csv('../data/promotion_final2.csv')

---

 전체 수신자 (100%)<br>
  └── 열람율: 쿠폰을 본 고객 비율 (비정상 흐름 포함)<br>
  │ &nbsp;&nbsp;&nbsp;└── 정상열람율: 그 중 정상 흐름으로 본 고객 비율<br>
  │              &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└──단계별전환율: 정상 열람자 중 완료까지 이어진 비율<br>
  └── 정상흐름비중: 받기 → 보기 → 완료 모두 정상인 비율<br>
  └── 비정상흐름비중: 이상 흐름 발생 비율<br>
  └── 바로완료비중: 보기 없이 바로 완료한 비율<br>
  └── 완료율: 최종적으로 완료한 전체 비율 (흐름 무관)<br>

## 검정 방법 설명

| 검정 방법 | 적용 기준 |
|---------|---------|
| **카이제곱 검정** | 두 범주형 변수 간 독립성 검정 (완료/미완료, 채널, 쿠폰 타입 등) |
| **Z-검정** | 두 그룹 간 비율 직접 비교 시 |
| **Cohen's h** | 두 비율 간 실질적 효과 크기 측정 시 |
| **사후검정(Bonferroni)** | 3개 이상 그룹 비교 후 어느 쌍이 다른지 확인 시 |
| **Fisher's Exact** | 특정 조합처럼 셀 빈도가 작을 가능성이 있을 때 |
| **스피어만 상관분석** | 순서형/연속형 변수 간 단조적 관계 검정 (난이도, 채널 수 등) |
| **피어슨 상관분석** | 연속형 변수 간 선형 관계 확인 시 |


## 우선순위 & 검정 방법 (A/B/C 분류)

### A. 쿠폰 기준

| # | 검증 주제 | 검정 방법 |
|---|---------|---------|
| 1 | `discount`가 `bogo`보다 정상흐름비중이 유의미하게 높은가 | 카이제곱 + Z-검정 + Cohen's h |
| 2 | `bogo`가 `discount`보다 열람율이 유의미하게 높은가 | 카이제곱 + Z-검정 + Cohen's h |
| 3 | `discount`가 `bogo`보다 단계별 전환율이 유의미하게 높은가 | 카이제곱 + Z-검정 + Cohen's h |
| 4 | 난이도가 높을수록 정상흐름비중이 낮아지는가 | 스피어만 + 피어슨 상관분석 |
| 5 | 8개 오퍼 간 완료율 차이가 유의미한가 | 카이제곱 + 사후검정(Bonferroni) |


### B. 채널 기준

| # | 검증 주제 | 검정 방법 |
|---------|---------|---------
| 1 | 채널별 열람율 차이가 유의미한가 | 카이제곱 + 사후검정(Bonferroni) |
| 2 | 채널별 정상흐름비중 차이가 유의미한가 | 카이제곱 + 사후검정(Bonferroni) |
| 3 | 채널별 단계별 전환율 차이가 유의미한가 | 카이제곱 + 사후검정(Bonferroni) |


### C. 채널 조합 기준

| # | 검증 주제 | 검정 방법 |
|---|---------|---------|
| 1 | 채널 조합별 정상흐름비중 차이가 유의미한가 | 카이제곱 + 사후검정(Bonferroni) |
| 2 | `social` 포함 여부에 따라 열람율이 유의미하게 다른가 | 카이제곱 + Z-검정 + Cohen's h |
| 3 | 채널 수가 많을수록 정상흐름비중이 높아지는가 | 스피어만 + 피어슨 상관분석 |
| 4 | 채널 조합별 열람율 차이가 유의미한가 | 카이제곱 + 사후검정(Bonferroni) |

---

In [4]:
bd_df = final[final['offer_type'].isin(['bogo', 'discount'])].copy()

# 1. 쿠폰

In [5]:
# 데이터 준비
bd_df['is_normal_flow'] = (bd_df['flow_type'] == 1).astype(int)
bd_df['is_direct'] = (bd_df['flow_type'] == 2).astype(int)

bogo_df = bd_df[bd_df['offer_type'] == 'bogo']
discount_df = bd_df[bd_df['offer_type'] == 'discount']
n_bogo = len(bogo_df)
n_discount = len(discount_df)

# ================================================================
# A-1. discount가 bogo보다 정상흐름비중이 유의미하게 높은가
# ================================================================
print("=" * 60)
print("A-1. discount vs bogo 정상흐름비중 비교")
print("=" * 60)

# 카이제곱
ct = pd.DataFrame({
    'normal': [discount_df['is_normal_flow'].sum(), bogo_df['is_normal_flow'].sum()],
    'not_normal': [len(discount_df) - discount_df['is_normal_flow'].sum(),
                   len(bogo_df) - bogo_df['is_normal_flow'].sum()]
}, index=['discount', 'bogo'])
chi2, p, dof, expected = chi2_contingency(ct)
print(f"\n[카이제곱 검정]")
print(f"Chi2: {chi2:.4f}, p-value: {p:.4f}, dof: {dof}")

# Z-검정 (단측: discount > bogo)
count = np.array([discount_df['is_normal_flow'].sum(), bogo_df['is_normal_flow'].sum()])
nobs = np.array([n_discount, n_bogo])
z_stat, p_val = proportions_ztest(count, nobs, alternative='larger')
print(f"\n[Z-검정 (단측: discount > bogo)]")
print(f"Z-stat: {z_stat:.4f}, p-value: {p_val:.4f}")

# Cohen's h
p1 = discount_df['is_normal_flow'].sum() / n_discount
p2 = bogo_df['is_normal_flow'].sum() / n_bogo
cohens_h = 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))
print(f"\n[Cohen's h] {cohens_h:.4f} (|h| < 0.2 작음 / 0.2~0.5 중간 / > 0.5 큼)")

# ================================================================
# A-2. bogo가 discount보다 열람율이 유의미하게 높은가
# ================================================================
print("\n" + "=" * 60)
print("A-2. bogo vs discount 열람율 비교")
print("=" * 60)

ct = pd.DataFrame({
    'viewed': [bogo_df['is_viewed'].sum(), discount_df['is_viewed'].sum()],
    'not_viewed': [len(bogo_df) - bogo_df['is_viewed'].sum(),
                   len(discount_df) - discount_df['is_viewed'].sum()]
}, index=['bogo', 'discount'])
chi2, p, dof, expected = chi2_contingency(ct)
print(f"\n[카이제곱 검정]")
print(f"Chi2: {chi2:.4f}, p-value: {p:.4f}, dof: {dof}")

count = np.array([bogo_df['is_viewed'].sum(), discount_df['is_viewed'].sum()])
nobs = np.array([n_bogo, n_discount])
z_stat, p_val = proportions_ztest(count, nobs, alternative='larger')
print(f"\n[Z-검정 (단측: bogo > discount)]")
print(f"Z-stat: {z_stat:.4f}, p-value: {p_val:.4f}")

p1 = bogo_df['is_viewed'].sum() / n_bogo
p2 = discount_df['is_viewed'].sum() / n_discount
cohens_h = 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))
print(f"\n[Cohen's h] {cohens_h:.4f}")

# ================================================================
# A-3. discount가 bogo보다 단계별 전환율이 유의미하게 높은가
# ================================================================
print("\n" + "=" * 60)
print("A-3. discount vs bogo 단계별 전환율 비교")
print("=" * 60)

viewed_bogo = bogo_df[bogo_df['flow_type'].isin([1, 3])]
viewed_discount = discount_df[discount_df['flow_type'].isin([1, 3])]
normal_bogo = bogo_df[bogo_df['flow_type'] == 1]
normal_discount = discount_df[discount_df['flow_type'] == 1]

ct = pd.DataFrame({
    'normal': [len(normal_discount), len(normal_bogo)],
    'not_normal': [len(viewed_discount) - len(normal_discount),
                   len(viewed_bogo) - len(normal_bogo)]
}, index=['discount', 'bogo'])
chi2, p, dof, expected = chi2_contingency(ct)
print(f"\n[카이제곱 검정]")
print(f"Chi2: {chi2:.4f}, p-value: {p:.4f}, dof: {dof}")

count = np.array([len(normal_discount), len(normal_bogo)])
nobs = np.array([len(viewed_discount), len(viewed_bogo)])
z_stat, p_val = proportions_ztest(count, nobs, alternative='larger')
print(f"\n[Z-검정 (단측: discount > bogo)]")
print(f"Z-stat: {z_stat:.4f}, p-value: {p_val:.4f}")

p1 = len(normal_discount) / len(viewed_discount)
p2 = len(normal_bogo) / len(viewed_bogo)
cohens_h = 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))
print(f"\n[Cohen's h] {cohens_h:.4f}")

# ================================================================
# A-4. 난이도가 높을수록 정상흐름비중이 낮아지는가
# ================================================================
print("\n" + "=" * 60)
print("A-4. 난이도 vs 정상흐름비중 상관분석")
print("=" * 60)

difficulty_normal = bd_df.groupby('difficulty').agg(
    total=('is_normal_flow', 'count'),
    normal=('is_normal_flow', 'sum')
).reset_index()
difficulty_normal['normal_rate'] = difficulty_normal['normal'] / difficulty_normal['total']
print(f"\n난이도별 정상흐름비중:")
print(difficulty_normal[['difficulty', 'normal_rate']].to_string(index=False))

spearman_corr, spearman_p = stats.spearmanr(
    difficulty_normal['difficulty'],
    difficulty_normal['normal_rate']
)
print(f"\n[스피어만 상관분석]")
print(f"상관계수: {spearman_corr:.4f}, p-value: {spearman_p:.4f}")

pearson_corr, pearson_p = stats.pearsonr(
    difficulty_normal['difficulty'],
    difficulty_normal['normal_rate']
)
print(f"\n[피어슨 상관분석]")
print(f"상관계수: {pearson_corr:.4f}, p-value: {pearson_p:.4f}")

# ================================================================
# A-5. 8개 오퍼 간 완료율 차이가 유의미한가
# ================================================================
print("\n" + "=" * 60)
print("A-5. 8개 오퍼 간 완료율 차이")
print("=" * 60)

offer_ids = bd_df['offer_id'].unique()
combo_data = {
    'offer_id': offer_ids,
    'completed': [bd_df[bd_df['offer_id'] == o]['is_completed'].sum() for o in offer_ids],
    'not_completed': [len(bd_df[bd_df['offer_id'] == o]) - 
                      bd_df[bd_df['offer_id'] == o]['is_completed'].sum() for o in offer_ids]
}
contingency_offer = pd.DataFrame(combo_data).set_index('offer_id')
chi2, p, dof, expected = chi2_contingency(contingency_offer)
print(f"\n[카이제곱 검정]")
print(f"Chi2: {chi2:.4f}, p-value: {p:.4f}, dof: {dof}")

if p < 0.05:
    print(f"\n[사후검정 - Bonferroni 쌍별 비교]")
    pairs_offer = [(a, b) for i, a in enumerate(offer_ids) for b in offer_ids[i+1:]]
    n_comparisons = len(pairs_offer)
    results = []
    for a, b in pairs_offer:
        df_a = bd_df[bd_df['offer_id'] == a]
        df_b = bd_df[bd_df['offer_id'] == b]
        ct = pd.DataFrame({
            'completed': [df_a['is_completed'].sum(), df_b['is_completed'].sum()],
            'not_completed': [len(df_a) - df_a['is_completed'].sum(),
                              len(df_b) - df_b['is_completed'].sum()]
        }, index=[a, b])
        chi2_pair, p_pair, _, _ = chi2_contingency(ct)
        p_adjusted = min(p_pair * n_comparisons, 1.0)
        results.append({
            'offer_a': a, 'offer_b': b,
            'p_value': round(p_pair, 4),
            'p_adjusted': round(p_adjusted, 4),
            'significant': '✅' if p_adjusted < 0.05 else '❌'
        })
    print(pd.DataFrame(results).sort_values('p_adjusted').to_string(index=False))

#

A-1. discount vs bogo 정상흐름비중 비교

[카이제곱 검정]
Chi2: 129.8274, p-value: 0.0000, dof: 1

[Z-검정 (단측: discount > bogo)]
Z-stat: 11.4025, p-value: 0.0000

[Cohen's h] 0.0923 (|h| < 0.2 작음 / 0.2~0.5 중간 / > 0.5 큼)

A-2. bogo vs discount 열람율 비교

[카이제곱 검정]
Chi2: 1499.3103, p-value: 0.0000, dof: 1

[Z-검정 (단측: bogo > discount)]
Z-stat: 38.7305, p-value: 0.0000

[Cohen's h] 0.3165

A-3. discount vs bogo 단계별 전환율 비교

[카이제곱 검정]
Chi2: 1057.9216, p-value: 0.0000, dof: 1

[Z-검정 (단측: discount > bogo)]
Z-stat: 32.5355, p-value: 0.0000

[Cohen's h] 0.3178

A-4. 난이도 vs 정상흐름비중 상관분석

난이도별 정상흐름비중:
 difficulty  normal_rate
          5     0.368573
          7     0.568663
         10     0.393668
         20     0.169536

[스피어만 상관분석]
상관계수: -0.4000, p-value: 0.6000

[피어슨 상관분석]
상관계수: -0.7977, p-value: 0.2023

A-5. 8개 오퍼 간 완료율 차이

[카이제곱 검정]
Chi2: 2046.2741, p-value: 0.0000, dof: 7

[사후검정 - Bonferroni 쌍별 비교]
         offer_a          offer_b  p_value  p_adjusted significant
      bogo_5_5_5 discount_10_2_10   0.0000  

---

# 2. 채널별

In [12]:
# 데이터 준비
web_df = bd_df[bd_df['web'] == 1]
email_df = bd_df[bd_df['email'] == 1]
mobile_df = bd_df[bd_df['mobile'] == 1]
social_df = bd_df[bd_df['social'] == 1]

channel_dfs = {'web': web_df, 'email': email_df, 'mobile': mobile_df, 'social': social_df}
channels = ['web', 'email', 'mobile', 'social']
pairs = [(a, b) for i, a in enumerate(channels) for b in channels[i+1:]]
n_comparisons = len(pairs)

# ================================================================
# B-1. 채널별 열람율 차이가 유의미한가
# ================================================================
print("=" * 60)
print("B-1. 채널별 열람율 차이")
print("=" * 60)

viewed_data = pd.DataFrame({
    'viewed': [channel_dfs[c]['is_viewed'].sum() for c in channels],
    'not_viewed': [len(channel_dfs[c]) - channel_dfs[c]['is_viewed'].sum() for c in channels]
}, index=channels)
chi2, p, dof, expected = chi2_contingency(viewed_data)
print(f"\n[카이제곱 검정]")
print(f"Chi2: {chi2:.4f}, p-value: {p:.4f}, dof: {dof}")

if p < 0.05:
    print(f"\n[사후검정 - Bonferroni 쌍별 비교]")
    results = []
    for a, b in pairs:
        ct = pd.DataFrame({
            'viewed': [channel_dfs[a]['is_viewed'].sum(), channel_dfs[b]['is_viewed'].sum()],
            'not_viewed': [len(channel_dfs[a]) - channel_dfs[a]['is_viewed'].sum(),
                           len(channel_dfs[b]) - channel_dfs[b]['is_viewed'].sum()]
        }, index=[a, b])
        chi2_pair, p_pair, _, _ = chi2_contingency(ct)
        p_adjusted = min(p_pair * n_comparisons, 1.0)
        results.append({
            'channel_a': a, 'channel_b': b,
            'p_value': round(p_pair, 4),
            'p_adjusted': round(p_adjusted, 4),
            'significant': '✅' if p_adjusted < 0.05 else '❌'
        })
    print(pd.DataFrame(results).sort_values('p_adjusted').to_string(index=False))

# ================================================================
# B-2. 채널별 정상흐름비중 차이가 유의미한가
# ================================================================
print("\n" + "=" * 60)
print("B-2. 채널별 정상흐름비중 차이")
print("=" * 60)

normal_data = pd.DataFrame({
    'normal': [channel_dfs[c]['is_normal_flow'].sum() for c in channels],
    'not_normal': [len(channel_dfs[c]) - channel_dfs[c]['is_normal_flow'].sum() for c in channels]
}, index=channels)
chi2, p, dof, expected = chi2_contingency(normal_data)
print(f"\n[카이제곱 검정]")
print(f"Chi2: {chi2:.4f}, p-value: {p:.4f}, dof: {dof}")

if p < 0.05:
    print(f"\n[사후검정 - Bonferroni 쌍별 비교]")
    results = []
    for a, b in pairs:
        ct = pd.DataFrame({
            'normal': [channel_dfs[a]['is_normal_flow'].sum(), channel_dfs[b]['is_normal_flow'].sum()],
            'not_normal': [len(channel_dfs[a]) - channel_dfs[a]['is_normal_flow'].sum(),
                           len(channel_dfs[b]) - channel_dfs[b]['is_normal_flow'].sum()]
        }, index=[a, b])
        chi2_pair, p_pair, _, _ = chi2_contingency(ct)
        p_adjusted = min(p_pair * n_comparisons, 1.0)
        results.append({
            'channel_a': a, 'channel_b': b,
            'p_value': round(p_pair, 4),
            'p_adjusted': round(p_adjusted, 4),
            'significant': '✅' if p_adjusted < 0.05 else '❌'
        })
    print(pd.DataFrame(results).sort_values('p_adjusted').to_string(index=False))

# ================================================================
# B-3. 채널별 단계별 전환율 차이가 유의미한가
# ================================================================
print("\n" + "=" * 60)
print("B-3. 채널별 단계별 전환율 차이")
print("=" * 60)

viewed_dfs = {c: channel_dfs[c][channel_dfs[c]['flow_type'].isin([1, 3])] for c in channels}

conversion_data = pd.DataFrame({
    'normal': [viewed_dfs[c]['is_completed'].sum() for c in channels],
    'not_normal': [len(viewed_dfs[c]) - viewed_dfs[c]['is_completed'].sum() for c in channels]
}, index=channels)
chi2, p, dof, expected = chi2_contingency(conversion_data)
print(f"\n[카이제곱 검정]")
print(f"Chi2: {chi2:.4f}, p-value: {p:.4f}, dof: {dof}")

if p < 0.05:
    print(f"\n[사후검정 - Bonferroni 쌍별 비교]")
    results = []
    for a, b in pairs:
        ct = pd.DataFrame({
            'normal': [viewed_dfs[a]['is_completed'].sum(), viewed_dfs[b]['is_completed'].sum()],
            'not_normal': [len(viewed_dfs[a]) - viewed_dfs[a]['is_completed'].sum(),
                           len(viewed_dfs[b]) - viewed_dfs[b]['is_completed'].sum()]
        }, index=[a, b])
        chi2_pair, p_pair, _, _ = chi2_contingency(ct)
        p_adjusted = min(p_pair * n_comparisons, 1.0)
        results.append({
            'channel_a': a, 'channel_b': b,
            'p_value': round(p_pair, 4),
            'p_adjusted': round(p_adjusted, 4),
            'significant': '✅' if p_adjusted < 0.05 else '❌'
        })
    print(pd.DataFrame(results).sort_values('p_adjusted').to_string(index=False))

B-1. 채널별 열람율 차이

[카이제곱 검정]
Chi2: 6466.7315, p-value: 0.0000, dof: 3

[사후검정 - Bonferroni 쌍별 비교]
channel_a channel_b  p_value  p_adjusted significant
      web     email      0.0         0.0           ✅
      web    mobile      0.0         0.0           ✅
      web    social      0.0         0.0           ✅
    email    mobile      0.0         0.0           ✅
    email    social      0.0         0.0           ✅
   mobile    social      0.0         0.0           ✅

B-2. 채널별 정상흐름비중 차이

[카이제곱 검정]
Chi2: 817.0335, p-value: 0.0000, dof: 3

[사후검정 - Bonferroni 쌍별 비교]
channel_a channel_b  p_value  p_adjusted significant
      web    mobile    0.000      0.0000           ✅
      web    social    0.000      0.0000           ✅
    email    mobile    0.000      0.0000           ✅
    email    social    0.000      0.0000           ✅
   mobile    social    0.000      0.0000           ✅
      web     email    0.029      0.1738           ❌

B-3. 채널별 단계별 전환율 차이

[카이제곱 검정]
Chi2: 64.4978, p-value: 0.0000, d

---

# 3. 채널 조합별

In [13]:
# 데이터 준비
def get_channel_combo(row):
    channels = []
    for ch in ['web', 'email', 'mobile', 'social']:
        if row[ch] == 1:
            channels.append(ch)
    return '+'.join(channels) if channels else 'none'

bd_df['channel_combo'] = bd_df.apply(get_channel_combo, axis=1)
bd_df['channel_count'] = bd_df['web'] + bd_df['email'] + bd_df['mobile'] + bd_df['social']

social_included = bd_df[bd_df['social'] == 1]
social_excluded = bd_df[bd_df['social'] == 0]

combos = bd_df['channel_combo'].unique()
pairs_combo = [(a, b) for i, a in enumerate(combos) for b in combos[i+1:]]
n_comparisons = len(pairs_combo)

# ================================================================
# C-1. 채널 조합별 정상흐름비중 차이가 유의미한가
# ================================================================
print("=" * 60)
print("C-1. 채널 조합별 정상흐름비중 차이")
print("=" * 60)

normal_data = pd.DataFrame({
    'normal': [bd_df[bd_df['channel_combo'] == c]['is_normal_flow'].sum() for c in combos],
    'not_normal': [len(bd_df[bd_df['channel_combo'] == c]) - 
                   bd_df[bd_df['channel_combo'] == c]['is_normal_flow'].sum() for c in combos]
}, index=combos)
chi2, p, dof, expected = chi2_contingency(normal_data)
print(f"\n[카이제곱 검정]")
print(f"Chi2: {chi2:.4f}, p-value: {p:.4f}, dof: {dof}")

if p < 0.05:
    print(f"\n[사후검정 - Bonferroni 쌍별 비교]")
    results = []
    for a, b in pairs_combo:
        df_a = bd_df[bd_df['channel_combo'] == a]
        df_b = bd_df[bd_df['channel_combo'] == b]
        ct = pd.DataFrame({
            'normal': [df_a['is_normal_flow'].sum(), df_b['is_normal_flow'].sum()],
            'not_normal': [len(df_a) - df_a['is_normal_flow'].sum(),
                           len(df_b) - df_b['is_normal_flow'].sum()]
        }, index=[a, b])
        chi2_pair, p_pair, _, _ = chi2_contingency(ct)
        p_adjusted = min(p_pair * n_comparisons, 1.0)
        results.append({
            'combo_a': a, 'combo_b': b,
            'p_value': round(p_pair, 4),
            'p_adjusted': round(p_adjusted, 4),
            'significant': '✅' if p_adjusted < 0.05 else '❌'
        })
    print(pd.DataFrame(results).sort_values('p_adjusted').to_string(index=False))

# ================================================================
# C-2. social 포함 여부에 따라 열람율이 유의미하게 다른가
# ================================================================
print("\n" + "=" * 60)
print("C-2. social 포함 여부 vs 열람율")
print("=" * 60)

ct = pd.DataFrame({
    'viewed': [social_included['is_viewed'].sum(), social_excluded['is_viewed'].sum()],
    'not_viewed': [len(social_included) - social_included['is_viewed'].sum(),
                   len(social_excluded) - social_excluded['is_viewed'].sum()]
}, index=['social_included', 'social_excluded'])
chi2, p, dof, expected = chi2_contingency(ct)
print(f"\n[카이제곱 검정]")
print(f"Chi2: {chi2:.4f}, p-value: {p:.4f}, dof: {dof}")

count = np.array([social_included['is_viewed'].sum(), social_excluded['is_viewed'].sum()])
nobs = np.array([len(social_included), len(social_excluded)])
z_stat, p_val = proportions_ztest(count, nobs, alternative='larger')
print(f"\n[Z-검정 (단측: social 포함 > 미포함)]")
print(f"Z-stat: {z_stat:.4f}, p-value: {p_val:.4f}")

p1 = social_included['is_viewed'].sum() / len(social_included)
p2 = social_excluded['is_viewed'].sum() / len(social_excluded)
cohens_h = 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))
print(f"\n[Cohen's h] {cohens_h:.4f} (|h| < 0.2 작음 / 0.2~0.5 중간 / > 0.5 큼)")

# ================================================================
# C-3. 채널 수가 많을수록 정상흐름비중이 높아지는가
# ================================================================
print("\n" + "=" * 60)
print("C-3. 채널 수 vs 정상흐름비중 상관분석")
print("=" * 60)

channel_count_normal = bd_df.groupby('channel_count').agg(
    total=('is_normal_flow', 'count'),
    normal=('is_normal_flow', 'sum')
).reset_index()
channel_count_normal['normal_rate'] = channel_count_normal['normal'] / channel_count_normal['total']
print(f"\n채널 수별 정상흐름비중:")
print(channel_count_normal[['channel_count', 'normal_rate']].to_string(index=False))

spearman_corr, spearman_p = stats.spearmanr(
    channel_count_normal['channel_count'],
    channel_count_normal['normal_rate']
)
print(f"\n[스피어만 상관분석]")
print(f"상관계수: {spearman_corr:.4f}, p-value: {spearman_p:.4f}")

pearson_corr, pearson_p = stats.pearsonr(
    channel_count_normal['channel_count'],
    channel_count_normal['normal_rate']
)
print(f"\n[피어슨 상관분석]")
print(f"상관계수: {pearson_corr:.4f}, p-value: {pearson_p:.4f}")

# ================================================================
# C-4. 채널 조합별 열람율 차이가 유의미한가
# ================================================================
print("\n" + "=" * 60)
print("C-4. 채널 조합별 열람율 차이")
print("=" * 60)

viewed_data = pd.DataFrame({
    'viewed': [bd_df[bd_df['channel_combo'] == c]['is_viewed'].sum() for c in combos],
    'not_viewed': [len(bd_df[bd_df['channel_combo'] == c]) - 
                   bd_df[bd_df['channel_combo'] == c]['is_viewed'].sum() for c in combos]
}, index=combos)
chi2, p, dof, expected = chi2_contingency(viewed_data)
print(f"\n[카이제곱 검정]")
print(f"Chi2: {chi2:.4f}, p-value: {p:.4f}, dof: {dof}")

if p < 0.05:
    print(f"\n[사후검정 - Bonferroni 쌍별 비교]")
    results = []
    for a, b in pairs_combo:
        df_a = bd_df[bd_df['channel_combo'] == a]
        df_b = bd_df[bd_df['channel_combo'] == b]
        ct = pd.DataFrame({
            'viewed': [df_a['is_viewed'].sum(), df_b['is_viewed'].sum()],
            'not_viewed': [len(df_a) - df_a['is_viewed'].sum(),
                           len(df_b) - df_b['is_viewed'].sum()]
        }, index=[a, b])
        chi2_pair, p_pair, _, _ = chi2_contingency(ct)
        p_adjusted = min(p_pair * n_comparisons, 1.0)
        results.append({
            'combo_a': a, 'combo_b': b,
            'p_value': round(p_pair, 4),
            'p_adjusted': round(p_adjusted, 4),
            'significant': '✅' if p_adjusted < 0.05 else '❌'
        })
    print(pd.DataFrame(results).sort_values('p_adjusted').to_string(index=False))

C-1. 채널 조합별 정상흐름비중 차이

[카이제곱 검정]
Chi2: 4045.9296, p-value: 0.0000, dof: 3

[사후검정 - Bonferroni 쌍별 비교]
                combo_a             combo_b  p_value  p_adjusted significant
web+email+mobile+social    web+email+mobile      0.0         0.0           ✅
web+email+mobile+social           web+email      0.0         0.0           ✅
web+email+mobile+social email+mobile+social      0.0         0.0           ✅
       web+email+mobile           web+email      0.0         0.0           ✅
       web+email+mobile email+mobile+social      0.0         0.0           ✅
              web+email email+mobile+social      0.0         0.0           ✅

C-2. social 포함 여부 vs 열람율

[카이제곱 검정]
Chi2: 17590.4096, p-value: 0.0000, dof: 1

[Z-검정 (단측: social 포함 > 미포함)]
Z-stat: 132.6387, p-value: 0.0000

[Cohen's h] 1.1407 (|h| < 0.2 작음 / 0.2~0.5 중간 / > 0.5 큼)

C-3. 채널 수 vs 정상흐름비중 상관분석

채널 수별 정상흐름비중:
 channel_count  normal_rate
             2     0.169536
             3     0.295642
             4     0.499128

[스피어만